## Lab 3 (Weeks 6–8): Support Case Analytics, Organizational Hierarchies, and Advanced Dimensional Modeling

> Before starting, review these resources:
> - `lab3_intro.md` — concepts covered in lecture
> - `hierarchy_intro.md` — how ragged hierarchies work and how to populate a bridge table (required for Task 5)

### What you are building
A brand-new business process — **customer support** — built on top of the dimensional infrastructure from Labs 1 and 2.

You will:
- Reuse `dim_account` and `dim_date` as conformed dimensions (no changes needed)
- Create `dim_support_case` (Type 1), `dim_user` (Type 2), `dim_contact` (Type 2)
- Build a bridge table for a ragged contact/manager organizational hierarchy
- Design and load `fact_support_case`

### Naming convention
All table names must end with your `_LAST_FI` suffix.
Save this notebook as `LAST_FI_LAB3` in the `MODELED` schema (e.g., `CLARK_C_LAB3`).

---
## Setup

In [ ]:
USE DATABASE SNOWBEARAIR_DB;
USE SCHEMA MODELED;
USE WAREHOUSE RAW_WH;

---
## Explore the Source Tables

Before designing anything, run these to understand the data. Pay attention to:
- What fields does `support_case` have? What are its date fields?
- How does `contact` relate to `salesforce_user`?
- What is `manager_id` in `contact` — does it point back to `contact`?

In [ ]:
SELECT * FROM SNOWBEARAIR_DB.PUBLIC.support_case     LIMIT 10;
-- SELECT * FROM SNOWBEARAIR_DB.PUBLIC.contact         LIMIT 10;
-- SELECT * FROM SNOWBEARAIR_DB.PUBLIC.salesforce_user LIMIT 10;

---
## Task 1: Declare the Business Process and Grain

Before writing any SQL, answer these questions:
1. What business process are we measuring?
2. What does one row in `fact_support_case` represent?

**Business Process:**
*(write your answer here)*

**Grain Statement:**
*(write your answer here — e.g., "One row per...")*

---
# Part 1: Design — Create Your Tables

**Build order (dependencies matter):**
1. `dim_support_case` (Type 1)
2. `dim_user` (Type 2)
3. `dim_contact` (Type 2)
4. `dim_management_hierarchy_bridge`
5. `fact_support_case`

**Conformed dimensions from prior labs** — use these directly, no changes needed:
- `dim_account_last_fi` (from Lab 2, Type 2)
- `dim_date_last_fi` (from Lab 2, Type 0)

---
### Task 2: Create dim_support_case (Type 1)

Create `dim_support_case_last_fi`

**Requirements:**
- `support_case_key` (AUTOINCREMENT)
- `case_id` natural key
- Descriptive attributes: case type, case status, case priority
- **Type 1 SCD** — no effective dates or current flag needed

**Why Type 1?** Status and priority reflect the current operational state. Analysts want to see whether a case is currently Open or Closed — not a history of every status transition.

In [ ]:
-- Task 2: Create dim_support_case_last_fi (Type 1)

CREATE OR REPLACE TABLE dim_support_case_last_fi (
    -- your columns
);

---
### Task 3: Create dim_user (Type 2)

Create `dim_user_last_fi`

**Requirements:**
- `user_key` (AUTOINCREMENT)
- `user_id` natural key
- Descriptive attributes: full name, title, department, role, active indicator
- **Type 2 SCD**: `effective_start_date`, `effective_end_date` (default `'9999-12-31'`), `current_flag`

**Why Type 2?** When a support agent changes departments or roles, historical cases must still show the correct organizational context at the time they were worked.

In [ ]:
-- Task 3: Create dim_user_last_fi (Type 2)

CREATE OR REPLACE TABLE dim_user_last_fi (
    -- your columns
);

---
### Task 4: Create dim_contact (Type 2)

Create `dim_contact_last_fi`

**Requirements:**
- `contact_key` (AUTOINCREMENT)
- `contact_id` natural key
- Descriptive attributes: first name, last name, email, title, department
- `manager_contact_id` — the **natural key** of this contact's manager (used for the hierarchy in Task 5)
- **Type 2 SCD**: `effective_start_date`, `effective_end_date`, `current_flag`

In [ ]:
-- Task 4: Create dim_contact_last_fi (Type 2)

CREATE OR REPLACE TABLE dim_contact_last_fi (
    -- your columns
);

---
### Task 5: Create dim_management_hierarchy_bridge

Create `dim_management_hierarchy_bridge_last_fi`

This bridge table stores **every subordinate-ancestor pair** in the org chart, regardless of depth. That's what makes it handle a ragged (variable-depth) hierarchy.

**Requirements:**
- `contact_key` — the subordinate (FK to `dim_contact_last_fi`)
- `manager_contact_key` — any ancestor at any level above (FK to `dim_contact_last_fi`)
- `levels_between` — 0 = self, 1 = direct manager, 2 = skip-level, etc.
- `top_flag` — TRUE if the manager_contact_key row is the top of the hierarchy
- `effective_start_date`, `effective_end_date`, `current_flag`

**Why a bridge table?** Not all contacts have the same number of reporting levels. A bridge table stores all relationships explicitly — no fixed-depth columns needed.

In [ ]:
-- Task 5: Create dim_management_hierarchy_bridge_last_fi

CREATE OR REPLACE TABLE dim_management_hierarchy_bridge_last_fi (
    -- your columns
);

---
### Task 6: Create the Fact Table

Create `fact_support_case_last_fi`

**Requirements:**
- `case_fact_key` (AUTOINCREMENT PK)
- Foreign keys: `account_key`, `support_case_key`, `user_key`, `contact_key`
- Role-playing date keys: `opened_date_key` and `closed_date_key` — both reference `dim_date_last_fi`
- Measure: `resolution_days` (INTEGER, NULL if case still open)

> The fact table must contain **only surrogate keys and measures** — no descriptive text, no natural identifiers.

In [ ]:
-- Task 6: Create fact_support_case_last_fi

CREATE OR REPLACE TABLE fact_support_case_last_fi (
    -- your columns
);

---
# Part 2: Load Data

**Load order:** dimensions → bridge → fact

---
### Load dim_support_case (Type 1)

In [ ]:
-- Load dim_support_case_last_fi

INSERT INTO dim_support_case_last_fi (
    -- your columns
)
SELECT
    -- your source columns
FROM SNOWBEARAIR_DB.PUBLIC.support_case;

---
### Load dim_user (Type 2 initial load)

Set `effective_start_date = CURRENT_DATE`, `effective_end_date = '9999-12-31'`, `current_flag = TRUE` for all rows on initial load.

In [ ]:
-- Load dim_user_last_fi (Type 2 initial load)

INSERT INTO dim_user_last_fi (
    -- your columns
)
SELECT
    -- your source columns
FROM SNOWBEARAIR_DB.PUBLIC.salesforce_user;

---
### Load dim_contact (Type 2 initial load)

In [ ]:
-- Load dim_contact_last_fi (Type 2 initial load)

INSERT INTO dim_contact_last_fi (
    -- your columns
)
SELECT
    -- your source columns
FROM SNOWBEARAIR_DB.PUBLIC.contact;

---
### Load dim_management_hierarchy_bridge

The bridge needs one row per subordinate-ancestor pair:
- A row for each contact pointing to themselves (levels_between = 0)
- A row for each contact pointing to their direct manager (levels_between = 1)

**Join pattern:** self-join `dim_contact_last_fi` on `manager_contact_id = contact_id`.

In [ ]:
-- Load dim_management_hierarchy_bridge_last_fi
-- Two-step approach: insert self rows first (levels_between = 0),
-- then insert direct manager rows (levels_between = 1) using a self-join on dim_contact.

INSERT INTO dim_management_hierarchy_bridge_last_fi (
    -- your columns
)
SELECT
    -- your expressions
FROM dim_contact_last_fi
-- your WHERE / JOIN conditions
;

---
### Load fact_support_case

Join through dimension tables to resolve surrogate keys.

- Use the date key resolution pattern from Lab 2 for `opened_date_key` and `closed_date_key`
- Calculate `resolution_days` from the open and close date columns (NULL if the case is still open)
- When joining Type 2 dimensions, filter to the current row

In [ ]:
-- Load fact_support_case_last_fi
-- Join through dimension tables to resolve surrogate keys.
-- Use the date key resolution and role-playing join pattern from Lab 2.

INSERT INTO fact_support_case_last_fi (
    -- your columns
)
SELECT
    -- your expressions
FROM SNOWBEARAIR_DB.PUBLIC.support_case src
-- your JOINs
;

---
## Verify Your Work

In [ ]:
-- Row counts
SELECT 'dim_support_case'                   AS table_name, COUNT(*) AS row_count FROM dim_support_case_last_fi
UNION ALL SELECT 'dim_user',                              COUNT(*)              FROM dim_user_last_fi
UNION ALL SELECT 'dim_contact',                           COUNT(*)              FROM dim_contact_last_fi
UNION ALL SELECT 'dim_management_hierarchy_bridge',       COUNT(*)              FROM dim_management_hierarchy_bridge_last_fi
UNION ALL SELECT 'fact_support_case',                     COUNT(*)              FROM fact_support_case_last_fi;

-- Cases by priority and status
-- SELECT sc.case_priority, sc.case_status, COUNT(*), AVG(f.resolution_days)
-- FROM fact_support_case_last_fi f
-- JOIN dim_support_case_last_fi sc ON sc.support_case_key = f.support_case_key
-- GROUP BY sc.case_priority, sc.case_status;

-- Roll up cases to any manager level using bridge table
-- SELECT mgr.first_name || ' ' || mgr.last_name AS manager, b.levels_between, COUNT(*) AS cases
-- FROM fact_support_case_last_fi f
-- JOIN dim_contact_last_fi c     ON c.contact_key = f.contact_key
-- JOIN dim_management_hierarchy_bridge_last_fi b ON b.contact_key = c.contact_key AND b.current_flag = TRUE
-- JOIN dim_contact_last_fi mgr   ON mgr.contact_key = b.manager_contact_key AND mgr.current_flag = TRUE
-- GROUP BY mgr.first_name, mgr.last_name, b.levels_between
-- ORDER BY manager, b.levels_between;

---
## Deliverables Checklist

- [ ] Notebook saved as `LAST_FI_LAB3` in the `MODELED` schema
- [ ] All table names include your `_LAST_FI` suffix
- [ ] Task 1 grain answer written in the markdown cell above
- [ ] All 5 DDL cells run successfully (Tasks 2–6)
- [ ] All load cells run without errors
- [ ] Bridge table has at least 2× rows vs contact count (self + manager rows)
- [ ] Row counts verified
- [ ] Star schema diagram uploaded **or** full DDL written (either/or)
- [ ] Written explanation in Canvas covering:
  - Fact grain justification
  - SCD strategy rationale for each dimension
  - Ragged hierarchy design: why a bridge table and how you'd query it
  - At least one modeling tradeoff you considered